In [1]:
import os
cwd = os.getcwd()
print("Current Working Directory:", cwd)

Current Working Directory: /projectnb/cs598/students/achetia


In [ ]:
from datasets import load_dataset
import io
import matplotlib.pyplot as plt
import PIL
from PIL import Image

#dataset = load_dataset("array/SAT", batch_size=128, cache_dir="/projectnb/cs598/students/achetia")
# dataset should have a training and validation key

example = dataset['train'][15] # example 10th item

image = example['image'] # this is a list of images. Some questions are on one image, and some on 2 images
question = example['question']
answer_choices = example['choices']
correct_answer = example['answer']

print(f"Question: {question}")
for idx, choice in enumerate(answer_choices):
    print(f"{idx + 1}: {choice}")

print(f"Correct Answer: {correct_answer}")

image.show()



In [ ]:
# %pip install nvidia-pyindex

In [ ]:
# %pip install nvidia-nccl

In [ ]:
# !nvidia-smi

In [ ]:
# !nvcc --version

In [2]:
import torch
import transformers
import torch.nn as nn
from transformers import AutoProcessor, AutoModelForCausalLM, TrainingArguments, AutoTokenizer
#from janus.models import VLChatProcessor, MultiModalityCausalLM
from transformers import LlavaForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import os
import json
from dataclasses import dataclass
from typing import Dict, List, Optional, Union
import PIL
from PIL import Image
import io
import numpy as np
import warnings
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import base64
from accelerate import Accelerator
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

warnings.simplefilter('ignore')

/projectnb/cs598/students/achetia/venvs/AJ/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

In [ ]:
# train_dataset = load_dataset("yangfu2/CVBench_Train")
# test_dataset = load_dataset("yangfu2/CVBench_Test")

# print(f"Train dataset size: {len(train_dataset['train'])}")
# print(f"Test dataset size: {len(test_dataset['train'])}")

In [ ]:
# def process_dataset(dataset, split, output_dir):
#     data = []
#     for item in dataset:
#         buffered = BytesIO()
#         item['image'].save(buffered, format="JPEG")
#         image_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
        
#         conversation = [
#             {"from": "human", "value": f"<image>\n{item['question']}"},
#             {"from": "gpt", "value": item['answer']}
#         ]
#         entry = {
#             "id": f"{split}_{item['idx']}",
#             "image": image_base64,  # Use the Base64 encoded string
#             "conversations": conversation
#         }
#         data.append(entry)
    
#     output_file = os.path.join(output_dir, f"{split}.json")
#     with open(output_file, 'w') as f:
#         json.dump(data, f, indent=2)


In [ ]:
# output_dir = "cvbench_llava_format"
# os.makedirs(output_dir, exist_ok=True)

# process_dataset(train_dataset['train'], "train", output_dir)
# process_dataset(test_dataset['train'], "test", output_dir)

In [4]:
# model_path = "deepseek-ai/Janus-Pro-7B"
model_path = "liuhaotian/llava-v1.5-7b"
cache_dir = "/projectnb/cs598/students/achetia/model/LlaVa"
output_d = "./LLAVA-lora-finetuned"
dataset_name = "array/SAT"

In [5]:
lora_config = LoraConfig(
    r=16,                       
    lora_alpha=16,              
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [6]:
print("Loading model and processor...")
#processor = AutoProcessor.from_pretrained(model_path, cache_dir=cache_dir)
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = LlavaForConditionalGeneration.from_pretrained(
    model_path, 
    trust_remote_code=True, 
    cache_dir=cache_dir
)
model = model.to(torch.float16)

# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs!")
#     # Wrap model with DataParallel
#     vl_gpt = nn.DataParallel(vl_gpt, device_ids=[0, 1])

model = model.cuda().eval()

Loading model and processor...


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  6.38it/s]
Some weights of LlavaForConditionalGeneration were not initialized from the model checkpoint at liuhaotian/llava-v1.5-7b and are newly initialized: ['model.language_model.lm_head.weight', 'model.language_model.model.embed_tokens.weight', 'model.language_model.model.layers.0.input_layernorm.weight', 'model.language_model.model.layers.0.mlp.down_proj.weight', 'model.language_model.model.layers.0.mlp.gate_proj.weight', 'model.language_model.model.layers.0.mlp.up_proj.weight', 'model.language_model.model.layers.0.post_attention_layernorm.weight', 'model.language_model.model.layers.0.self_attn.k_proj.weight', 'model.language_model.model.layers.0.self_attn.o_proj.weight', 'model.language_model.model.layers.0.self_attn.q_proj.weight', 'model.language_model.model.layers.0.self_attn.v_proj.weight', 'model.language_model.

In [7]:
import GPUtil

# Get all GPUs
gpus = GPUtil.getGPUs()

# Print details for each GPU
for gpu in gpus:
    print(f"GPU ID: {gpu.id}")
    print(f"GPU Name: {gpu.name}")
    print(f"Total Memory: {gpu.memoryTotal} MB")
    print(f"Free Memory: {gpu.memoryFree} MB")
    print(f"Memory Utilization: {gpu.memoryUtil*100:.2f}%")
    print(f"GPU Load: {gpu.load*100:.2f}%")
    print("-" * 30)


GPU ID: 0
GPU Name: NVIDIA A40
Total Memory: 46068.0 MB
Free Memory: 25809.0 MB
Memory Utilization: 42.74%
GPU Load: 0.00%
------------------------------
GPU ID: 1
GPU Name: NVIDIA A40
Total Memory: 46068.0 MB
Free Memory: 31713.0 MB
Memory Utilization: 29.93%
GPU Load: 31.00%
------------------------------


In [8]:
# model = vl_gpt.to(torch.device("cuda:1"))
print("Preparing model for LoRA fine-tuning...")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Preparing model for LoRA fine-tuning...
trainable params: 42,336,256 || all params: 7,105,239,040 || trainable%: 0.5958456254836995


In [9]:

dataset = load_dataset(dataset_name, batch_size=128, cache_dir="/projectnb/cs598/students/achetia")
print("Dataset loaded")

Dataset loaded


In [10]:
# def preprocess_function(examples):
#     for example in examples:
#         image = [example["image"]]
            
#         question = example["question"]
#         answer_choices = example["choices"]
#         correct_answer = example["answer"]

#         prompt = f"""
#         Instructions: Answer the following question using the given options.
#         Enclose your answer in [].

#         Question: {question}

#         Options: {answer_choices}
#         """

#         response = f"[{correct_answer}]"
        
#         conversations = [
#             {
#                 "role": "<|User|>",
#                 "content": f"<image_placeholder>\n{prompt}",
#                 "images": image,
#             },
#             {"role": "<|Assistant|>", "content": response},
#         ]
    
#     pil_images = load_pil_images(conversations)
#     inputs = vl_chat_processor(
#         conversations=conversations, images=pil_images, force_batchify=True
#     ).to(vl_gpt.device)
    
#     labels = tokenizer(examples["answer"], padding="max_length", truncation=True, return_tensors="pt")["input_ids"]
    
#     return {"inputs_embeds": inputs["inputs_embeds"], "labels": labels}
@dataclass
class SATDataCollator:
    max_length: int = 512
    
    def __call__(self, examples):
        batch_inputs = []
        batch_labels = []
        batch_attention_masks = []
        
        for example in examples:
            image = [im_bytes for im_bytes in example["image_bytes"]]
            
            question = example["question"]
            answer_choices = example["answers"]
            correct_answer = example["correct_answer"]
            
            prompt = f"""
            Instructions: Answer the following question using the given options.
            Enclose your answer in [].
            
            Question: {question}
            
            Options: {answer_choices}
            """
            
            response = f"[{correct_answer}]"
            
            text = f"<image>\n{prompt}"
            
            inputs = tokenizer(
                text=text,  
                return_tensors="pt", 
                padding="max_length",
                max_length=self.max_length,
                truncation=True
            )
            
            target_inputs = tokenizer(
                text=response,
                return_tensors="pt",
                padding="max_length",
                max_length=self.max_length,
                truncation=True
            )
            
            input_ids = inputs.input_ids[0]
            attention_mask = inputs.attention_mask[0]
            
            labels = target_inputs.input_ids[0]
            
            batch_inputs.append(input_ids)
            batch_labels.append(labels)
            batch_attention_masks.append(attention_mask)
        
        max_len = max(len(ids) for ids in batch_inputs)
        
        padded_inputs = torch.ones((len(batch_inputs), max_len), dtype=torch.long) * tokenizer.pad_token_id
        padded_labels = torch.ones((len(batch_labels), max_len), dtype=torch.long) * -100
        attention_masks = torch.zeros((len(batch_inputs), max_len), dtype=torch.long)
        
        for i, (input_ids, label_ids, attn_mask) in enumerate(zip(batch_inputs, batch_labels, batch_attention_masks)):
            input_len = len(input_ids)
            padded_inputs[i, :input_len] = input_ids
            padded_labels[i, :input_len] = label_ids
            attention_masks[i, :input_len] = attn_mask[:input_len]
        
        return {
            "input_ids": padded_inputs,
            "attention_mask": attention_masks,
            "labels": padded_labels,
            "pixel_values": self._process_images(examples) 
        }
    
    def _process_images(self, examples):
        processed_images = []
        for example in examples:
            image = example.get("image")
            if image:
                processed_images.append(image)
        return torch.tensor(processed_images) if processed_images else None
    

In [11]:
data_collator = SATDataCollator( max_length=512)
accelerator = Accelerator()

In [ ]:
# training_args = TrainingArguments(
#     output_dir=output_dir,
#     num_train_epochs=3,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     warmup_steps=100,
#     logging_steps=10,
#     save_steps=200,
#     fp16=True,  # Use mixed precision training
#     optim="adamw_torch",
#     report_to="tensorboard",
#     remove_unused_columns=False,
#     local_rank=-1,  # for distributed training
#     ddp_find_unused_parameters=False,
# )

In [ ]:
# trainer = transformers.Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     eval_dataset=dataset["validation"] if "validation" in dataset else None,
#     data_collator=data_collator,
# )

In [ ]:
# print("Starting training...")
# trainer.train()

In [12]:
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, 
    AdamW(model.parameters(), lr=2e-4),
    torch.utils.data.DataLoader(dataset["train"], batch_size=1, collate_fn=data_collator),
    torch.utils.data.DataLoader(dataset["validation"], batch_size=1, collate_fn=data_collator) if "validation" in dataset else None
)

In [13]:
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = 3 * num_update_steps_per_epoch
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=100, num_training_steps=num_training_steps
)

In [ ]:
save_dir = "./LLAVA-lora-finetuned"
os.makedirs(save_dir, exist_ok=True)

for epoch in range(1):
    model.train()
    for batch_idx, batch in enumerate(train_dataloader):
        with accelerator.accumulate(model):
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
        
        accelerator.print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item()}")
        
        if (batch_idx + 1) % 100 == 0 and accelerator.is_main_process:
            save_path = os.path.join(save_dir, f"epoch_{epoch}_batch_{batch_idx}.pt")
            accelerator.save_state(
                
                output_dir=save_dir,
                safe_serialization=True,
                weights_only=True
            )
    
    if eval_dataloader:
        model.eval()
        eval_loss = 0
        for batch in eval_dataloader:
            with torch.no_grad():
                outputs = model(**batch)
            eval_loss += outputs.loss.item()
        eval_loss /= len(eval_dataloader)
        accelerator.print(f"Epoch {epoch}, Eval Loss: {eval_loss}")
        
    if accelerator.is_main_process:
        accelerator.save(
            accelerator.unwrap_model(model).state_dict(),
            f"./LLAVA-lora-finetuned/epoch_{epoch}.pt"
        )

print("Training completed")

Epoch 0, Batch 0, Loss: 11.58250617980957
Epoch 0, Batch 1, Loss: 11.658019065856934
Epoch 0, Batch 2, Loss: 11.607669830322266
Epoch 0, Batch 3, Loss: 11.258770942687988
Epoch 0, Batch 4, Loss: 11.431425094604492
Epoch 0, Batch 5, Loss: 11.194581985473633
Epoch 0, Batch 6, Loss: 11.230779647827148
Epoch 0, Batch 7, Loss: 11.252827644348145
Epoch 0, Batch 8, Loss: 11.006174087524414
Epoch 0, Batch 9, Loss: 10.782219886779785
Epoch 0, Batch 10, Loss: 10.770833015441895
Epoch 0, Batch 11, Loss: 10.444374084472656
Epoch 0, Batch 12, Loss: 10.525257110595703
Epoch 0, Batch 13, Loss: 10.312263488769531
Epoch 0, Batch 14, Loss: 10.329615592956543
Epoch 0, Batch 15, Loss: 8.956964492797852
Epoch 0, Batch 16, Loss: 8.728291511535645
Epoch 0, Batch 17, Loss: 7.911925315856934
Epoch 0, Batch 18, Loss: 6.732865333557129
Epoch 0, Batch 19, Loss: 6.265673637390137
Epoch 0, Batch 20, Loss: 4.912250518798828
Epoch 0, Batch 21, Loss: 4.247880458831787
Epoch 0, Batch 22, Loss: 3.7404909133911133
Epoch 

Epoch 0, Batch 183, Loss: 0.9152628779411316
Epoch 0, Batch 184, Loss: 0.14750626683235168
Epoch 0, Batch 185, Loss: 0.14205746352672577
Epoch 0, Batch 186, Loss: 0.06121758371591568
Epoch 0, Batch 187, Loss: 0.09709522873163223
Epoch 0, Batch 188, Loss: 0.06919461488723755
Epoch 0, Batch 189, Loss: 0.08274492621421814
Epoch 0, Batch 190, Loss: 0.03393867611885071
Epoch 0, Batch 191, Loss: 0.07766148447990417
Epoch 0, Batch 192, Loss: 0.03180061653256416
Epoch 0, Batch 193, Loss: 0.040180351585149765
Epoch 0, Batch 194, Loss: 0.05013564974069595
Epoch 0, Batch 195, Loss: 0.04764460027217865
Epoch 0, Batch 196, Loss: 0.04566289111971855
Epoch 0, Batch 197, Loss: 0.04525650665163994
Epoch 0, Batch 198, Loss: 1.317917823791504
Epoch 0, Batch 199, Loss: 0.05268026143312454
Epoch 0, Batch 200, Loss: 0.09723688662052155
Epoch 0, Batch 201, Loss: 0.051219481974840164
Epoch 0, Batch 202, Loss: 0.04675723612308502
Epoch 0, Batch 203, Loss: 0.05521143972873688
Epoch 0, Batch 204, Loss: 0.0440000

Epoch 0, Batch 362, Loss: 0.060024626553058624
Epoch 0, Batch 363, Loss: 0.05124221369624138
Epoch 0, Batch 364, Loss: 0.04496306926012039
Epoch 0, Batch 365, Loss: 0.0898754671216011
Epoch 0, Batch 366, Loss: 0.046828679740428925
Epoch 0, Batch 367, Loss: 0.04561798274517059
Epoch 0, Batch 368, Loss: 0.03847438469529152
Epoch 0, Batch 369, Loss: 0.09138182550668716
Epoch 0, Batch 370, Loss: 0.05334581434726715
Epoch 0, Batch 371, Loss: 0.07124625891447067
Epoch 0, Batch 372, Loss: 0.05900021269917488
Epoch 0, Batch 373, Loss: 0.04505590721964836
Epoch 0, Batch 374, Loss: 0.049529533833265305
Epoch 0, Batch 375, Loss: 0.03362639248371124
Epoch 0, Batch 376, Loss: 0.04432573541998863
Epoch 0, Batch 377, Loss: 0.04836975038051605
Epoch 0, Batch 378, Loss: 0.037549939006567
Epoch 0, Batch 379, Loss: 0.02442755177617073
Epoch 0, Batch 380, Loss: 0.03022707626223564
Epoch 0, Batch 381, Loss: 0.044892922043800354
Epoch 0, Batch 382, Loss: 0.10128900408744812
Epoch 0, Batch 383, Loss: 0.09603

Epoch 0, Batch 541, Loss: 1.54685378074646
Epoch 0, Batch 542, Loss: 0.08732710778713226
Epoch 0, Batch 543, Loss: 0.10746252536773682
Epoch 0, Batch 544, Loss: 0.142866313457489
Epoch 0, Batch 545, Loss: 0.1297406554222107
Epoch 0, Batch 546, Loss: 0.09342632442712784
Epoch 0, Batch 547, Loss: 0.06279388815164566
Epoch 0, Batch 548, Loss: 0.9830740094184875
Epoch 0, Batch 549, Loss: 0.08404804766178131
Epoch 0, Batch 550, Loss: 0.07882262021303177
Epoch 0, Batch 551, Loss: 0.07093024253845215
Epoch 0, Batch 552, Loss: 0.07593435049057007
Epoch 0, Batch 553, Loss: 0.0580124706029892
Epoch 0, Batch 554, Loss: 0.06288617104291916
Epoch 0, Batch 555, Loss: 0.04225471615791321
Epoch 0, Batch 556, Loss: 0.057198233902454376
Epoch 0, Batch 557, Loss: 0.04552531614899635
Epoch 0, Batch 558, Loss: 0.045877642929553986
Epoch 0, Batch 559, Loss: 0.04318685457110405
Epoch 0, Batch 560, Loss: 0.04291723296046257
Epoch 0, Batch 561, Loss: 0.051736339926719666
Epoch 0, Batch 562, Loss: 0.03289281204

Epoch 0, Batch 720, Loss: 0.04987234249711037
Epoch 0, Batch 721, Loss: 0.048156484961509705
Epoch 0, Batch 722, Loss: 0.5990809798240662
Epoch 0, Batch 723, Loss: 0.08739504963159561
Epoch 0, Batch 724, Loss: 0.0454072542488575
Epoch 0, Batch 725, Loss: 0.04211452975869179
Epoch 0, Batch 726, Loss: 0.09376714378595352
Epoch 0, Batch 727, Loss: 0.055152058601379395
Epoch 0, Batch 728, Loss: 0.08417046815156937
Epoch 0, Batch 729, Loss: 0.061548128724098206
Epoch 0, Batch 730, Loss: 0.04441343992948532
Epoch 0, Batch 731, Loss: 0.03644164279103279
Epoch 0, Batch 732, Loss: 0.037130676209926605
Epoch 0, Batch 733, Loss: 0.039924897253513336
Epoch 0, Batch 734, Loss: 0.04144700616598129
Epoch 0, Batch 735, Loss: 0.03871224820613861
Epoch 0, Batch 736, Loss: 0.03802168741822243
Epoch 0, Batch 737, Loss: 0.017224621027708054
Epoch 0, Batch 738, Loss: 0.024124858900904655
Epoch 0, Batch 739, Loss: 1.259574294090271
Epoch 0, Batch 740, Loss: 0.046074990183115005
Epoch 0, Batch 741, Loss: 0.04

...

Epoch 0, Batch 28463, Loss: 0.022344691678881645
Epoch 0, Batch 28464, Loss: 0.03188551962375641
Epoch 0, Batch 28465, Loss: 0.43484827876091003
Epoch 0, Batch 28466, Loss: 0.04542013257741928
Epoch 0, Batch 28467, Loss: 0.04763169214129448
Epoch 0, Batch 28468, Loss: 0.0454728864133358
Epoch 0, Batch 28469, Loss: 0.44515618681907654
Epoch 0, Batch 28470, Loss: 0.2155582755804062
Epoch 0, Batch 28471, Loss: 0.5029703974723816
Epoch 0, Batch 28472, Loss: 0.04202074185013771
Epoch 0, Batch 28473, Loss: 0.03822601959109306
Epoch 0, Batch 28474, Loss: 0.038327526301145554
Epoch 0, Batch 28475, Loss: 0.03602436184883118
Epoch 0, Batch 28476, Loss: 0.046696074306964874
Epoch 0, Batch 28477, Loss: 0.04756559804081917
Epoch 0, Batch 28478, Loss: 0.030273647978901863
Epoch 0, Batch 28479, Loss: 0.035775523632764816
Epoch 0, Batch 28480, Loss: 0.44881752133369446
Epoch 0, Batch 28481, Loss: 0.05132284760475159
Epoch 0, Batch 28482, Loss: 0.04401931166648865
Epoch 0, Batch 28483, Loss: 0.04571346

Epoch 0, Batch 28634, Loss: 0.04046981781721115
Epoch 0, Batch 28635, Loss: 0.03658708557486534
Epoch 0, Batch 28636, Loss: 0.037803348153829575
Epoch 0, Batch 28637, Loss: 0.03577519208192825
Epoch 0, Batch 28638, Loss: 0.04441213980317116
Epoch 0, Batch 28639, Loss: 0.03775280341506004
Epoch 0, Batch 28640, Loss: 0.035996340215206146
Epoch 0, Batch 28641, Loss: 0.030186936259269714
Epoch 0, Batch 28642, Loss: 0.5139999389648438
Epoch 0, Batch 28643, Loss: 0.05946572124958038
Epoch 0, Batch 28644, Loss: 0.0586479976773262
Epoch 0, Batch 28645, Loss: 0.05862480029463768
Epoch 0, Batch 28646, Loss: 0.9989950656890869
Epoch 0, Batch 28647, Loss: 0.24986396729946136
Epoch 0, Batch 28648, Loss: 3.3687479496002197
Epoch 0, Batch 28649, Loss: 0.061142463237047195
Epoch 0, Batch 28650, Loss: 0.05601174011826515
Epoch 0, Batch 28651, Loss: 0.0701175108551979
Epoch 0, Batch 28652, Loss: 0.09494003653526306
Epoch 0, Batch 28653, Loss: 0.08129266649484634
Epoch 0, Batch 28654, Loss: 0.07631272822

Epoch 0, Batch 28805, Loss: 0.0407264307141304
Epoch 0, Batch 28806, Loss: 0.0404653362929821
Epoch 0, Batch 28807, Loss: 0.05246046930551529
Epoch 0, Batch 28808, Loss: 0.057076580822467804
Epoch 0, Batch 28809, Loss: 0.056500427424907684
Epoch 0, Batch 28810, Loss: 0.036847297102212906
Epoch 0, Batch 28811, Loss: 0.03876488283276558
Epoch 0, Batch 28812, Loss: 0.6458529233932495
Epoch 0, Batch 28813, Loss: 0.05022403597831726
Epoch 0, Batch 28814, Loss: 0.05243823677301407
Epoch 0, Batch 28815, Loss: 0.07804986089468002
Epoch 0, Batch 28816, Loss: 0.08949051052331924
Epoch 0, Batch 28817, Loss: 0.06805066764354706
Epoch 0, Batch 28818, Loss: 0.09556840360164642
Epoch 0, Batch 28819, Loss: 0.04811317101120949
Epoch 0, Batch 28820, Loss: 0.055282410234212875
Epoch 0, Batch 28821, Loss: 0.04090352728962898
Epoch 0, Batch 28822, Loss: 0.04763425141572952
Epoch 0, Batch 28823, Loss: 0.050221849232912064
Epoch 0, Batch 28824, Loss: 0.04234801232814789
Epoch 0, Batch 28825, Loss: 0.03573820

Epoch 0, Batch 28976, Loss: 0.0629236027598381
Epoch 0, Batch 28977, Loss: 0.034999698400497437
Epoch 0, Batch 28978, Loss: 0.06468605250120163
Epoch 0, Batch 28979, Loss: 0.03601609542965889
Epoch 0, Batch 28980, Loss: 0.05815158411860466
Epoch 0, Batch 28981, Loss: 0.03322295472025871
Epoch 0, Batch 28982, Loss: 0.035826291888952255
Epoch 0, Batch 28983, Loss: 0.3604380786418915
Epoch 0, Batch 28984, Loss: 0.03386824578046799
Epoch 0, Batch 28985, Loss: 0.059439510107040405
Epoch 0, Batch 28986, Loss: 0.03378767520189285
Epoch 0, Batch 28987, Loss: 0.07365676015615463
Epoch 0, Batch 28988, Loss: 0.047847338020801544
Epoch 0, Batch 28989, Loss: 0.07635550200939178
Epoch 0, Batch 28990, Loss: 0.030554093420505524
Epoch 0, Batch 28991, Loss: 0.04788096994161606
Epoch 0, Batch 28992, Loss: 0.026997331529855728
Epoch 0, Batch 28993, Loss: 0.04040505364537239
Epoch 0, Batch 28994, Loss: 0.040585000067949295
Epoch 0, Batch 28995, Loss: 0.03601987659931183
Epoch 0, Batch 28996, Loss: 0.03195

Epoch 0, Batch 29147, Loss: 0.05201256647706032
Epoch 0, Batch 29148, Loss: 0.0591474324464798
Epoch 0, Batch 29149, Loss: 0.059361714869737625
Epoch 0, Batch 29150, Loss: 0.06687946617603302
Epoch 0, Batch 29151, Loss: 0.059990379959344864
Epoch 0, Batch 29152, Loss: 0.05629008263349533
Epoch 0, Batch 29153, Loss: 0.0472102165222168
Epoch 0, Batch 29154, Loss: 0.31948795914649963
Epoch 0, Batch 29155, Loss: 0.10781192779541016
Epoch 0, Batch 29156, Loss: 0.1065669134259224
Epoch 0, Batch 29157, Loss: 0.10296310484409332
Epoch 0, Batch 29158, Loss: 0.08474437892436981
Epoch 0, Batch 29159, Loss: 0.06319722533226013
Epoch 0, Batch 29160, Loss: 0.07816416025161743
Epoch 0, Batch 29161, Loss: 0.03885304927825928
Epoch 0, Batch 29162, Loss: 0.04871264845132828
Epoch 0, Batch 29163, Loss: 0.03396257758140564
Epoch 0, Batch 29164, Loss: 0.03560156002640724
Epoch 0, Batch 29165, Loss: 0.04435061663389206
Epoch 0, Batch 29166, Loss: 0.03278430923819542
Epoch 0, Batch 29167, Loss: 0.03226422891

In [ ]:
from tqdm import tqdm
import re
from PIL import Image
import io

# Function to extract the answer from the prediction text
def extract_ans(pred_text):
    matches = re.findall(r"\[(.*?)\]", pred_text)
    return matches

# Modified prediction function for the 'array/SAT' dataset
def evaluate_on_validation(dataset, vl_chat_processor, vl_gpt, device='cuda'):
    predictions = []
    n_correct = 0
    total = len(dataset['validation'])
    
    for i in (pbar := tqdm(range(total))):  
        example = dataset['validation'][i]  

        image_bytes = example['image_bytes']
        images = image_bytes  # List of images

        question = example['question']
        answer_choices = example['answers']
        correct_answer = example['correct_answer']

        input_prompt = f'''
                        Instructions: You have to answer the following question using the given options.
                                      Enclose your answer in []. 
                                      For example, if the answer is yes, it must be formatted like [yes].

                        Question: {question}

                        Image(s): {len(images)} image(s) are provided. 
                        Your answer must contain just the options given below. Enclose the answer in brackets like [answer]. 
                        Options: {answer_choices}
                '''

        # Generate the prediction text using the model
        prediction_text = janus_pro_generate(
            vl_chat_processor=vl_chat_processor,
            vl_gpt=vl_gpt,
            input_text=input_prompt,
            input_image=images,
            device=device,
            output_mode="text",
            temperature=0.1,
            top_p=0.95,
            seed=42
        )
        
        # Extract the prediction answer
        prediction = extract_ans(prediction_text)

        predictions.append({
            "question": question,
            "answers": answer_choices,
            "pred_text": prediction_text,
            "prediction": prediction,
            "correct_answer": correct_answer
        })

        # Check if the prediction matches the correct answer
        try:
            if len(prediction) == 1:
                if prediction[0].strip().lower() == correct_answer.strip().lower():
                    n_correct += 1
        except:
            pass

        # Update the progress bar description with accuracy
        accuracy = (n_correct / (i + 1)) * 100
        pbar.set_description(f'Accuracy: {accuracy:.2f}% ({n_correct}/{i+1})')

    # Calculate final accuracy
    final_accuracy = (n_correct / total) * 100
    print(f"Final accuracy: {final_accuracy:.2f}% ({n_correct}/{total})")
    
    return predictions, final_accuracy

# Run evaluation on validation set
validation_predictions, validation_accuracy = evaluate_on_validation(dataset, vl_chat_processor, vl_gpt)

# Save results to file
import json
with open('validation_results.json', 'w') as f:
    json.dump({
        'predictions': validation_predictions,
        'accuracy': validation_accuracy
    }, f, indent=2)

print(f"Results saved to validation_results.json")


In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, LlavaForConditionalGeneration, AutoProcessor
from torchvision import transforms
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import os
from dataclasses import dataclass
from PIL import Image
from tqdm import tqdm
from accelerate import Accelerator
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import warnings

warnings.simplefilter('ignore')

# Configuration
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
model_path = "liuhaotian/llava-v1.5-7b"
cache_dir = "/projectnb/cs598/students/achetia/model/LlaVa"
output_dir = "./LLAVA-lora-clevr-finetuned"  
os.makedirs(output_dir, exist_ok=True)

# Enhanced LoRA Config for spatial reasoning
lora_config = LoraConfig(
    r=64,                        
    lora_alpha=32,               
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"  
    ],#,"gate_proj", "up_proj", "down_proj"
    lora_dropout=0.1,
    bias="lora_only",
    task_type="CAUSAL_LM",
    modules_to_save=["embed_tokens", "lm_head"]  
)

# Load model and tokenizer
print("Loading model and processor...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
#processor = AutoProcessor.from_pretrained(model_path, cache_dir=cache_dir)

model = LlavaForConditionalGeneration.from_pretrained(
    model_path, 
    trust_remote_code=True, 
    cache_dir=cache_dir
)
model = model.to(torch.float16)
model = model.cuda().eval()

# Prepare for LoRA training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load CLEVR dataset
print("Loading CLEVR dataset...")
clevr_dataset = load_dataset(
    "laion/clevr-webdataset",
    split="train[:300000]",
    cache_dir="/projectnb/cs598/students/achetia"
)

: 